In [ ]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone
from langchain_pinecone import PineconeVectorStore
import pinecone
# from langchain.document_loaders import PyPDFLoader, DirectoryLoaderpin 
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers
# from tqdm.autonotebook import tqdm




In [ ]:
# PINECONE_API_KEY = "pcsk_7JfyyM_H8UeRDuTfBPS32vTMNwBMc6pLCkYJQtHmWPnzVartGJRjeJFcXFSWeKMM2jJUeH"
# PINECONE_API_ENV = ""

In [ ]:
# Extract Data from PDF

def load_pdf(data):
    loader = DirectoryLoader(data,
                             glob="*.pdf",
                             loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents
                  
    


In [ ]:
extracted_data = load_pdf("data/")


In [ ]:
#Create text chunks - List of LangChain Document Object(1. page_content 2. Metadata)

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


In [ ]:
text_chunks = text_split(extracted_data)
print("Length of my chunk : " , len(text_chunks))

In [ ]:
#Download embedding model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [ ]:
embeddings = download_hugging_face_embeddings()

In [ ]:
# query_result = emebeddings.embed_query("Hello world")
# print("length", len(query_result))

In [ ]:
#Initialzing the pinecone
import os
# from langchain_pinecone import PineconeVectorStore
# PINECONE_API_KEY = ""
os.environ['PINECONE_API_KEY'] = "pcsk_7JfyyM_H8UeRDuTfBPS32vTMNwBMc6pLCkYJQtHmWPnzVartGJRjeJFcXFSWeKMM2jJUeH"
# os.environ['PINECONE_API_KEY']
index_name = "medical-chatbot"

# docsearch = PineconeVectorStore.from_texts([t.page_content for t in text_chunks], embeddings, index_name=index_name)

In [ ]:
# docsearch = Pinecone.from_existing_index(index_name, embeddings)
docsearch = PineconeVectorStore.from_existing_index(index_name, embeddings)
# query = "What are allergies"
# docs = docsearch.similarity_search(query, k=3)
# print("Result ", docs)


In [ ]:
Prompt_template = """ 
    Use the following pieces of information to answer the user's question.
    If you dont' know the answer, just say that you don't know, don't try to make up an answer.

    Context: {context}
    Question: {question}
    
    Only return the helpful answer below and nothing else.
    Helpufl answer:
    """

In [ ]:
PROMPT = PromptTemplate(template=Prompt_template, input_variables=["context", "question"])
chain_type_kwargs = {"prompt": PROMPT}

In [ ]:
llm = CTransformers(model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
                    model_type="llama",
                    config={
                            'max_new_tokens': 512,
                            'temperature':0.8})




In [ ]:
qa=RetrievalQA.from_chain_type(
    llm= llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={'k':2}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)

In [ ]:
while True:
    user_input=input(f"Input Prompt:")
    result=qa.invoke({"query":user_input})
    print("Response :", result["result"])
